# Import Environment variables

In [ ]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

In [ ]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

# Import library

In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import subprocess
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import my_function as mf

# Data's import

## Clinical data

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_eur_cohorte_raw_bcsc_mammorisk_prs_298_305_311.tsv"

df_eur = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

## BCSC v2 Model

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_test_eur_bcscRisk_recalib.tsv"

df_bcsc = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

## Mammorisk Model

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_test_eur_mammorisk_recalib.tsv"

df_mammorisk = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

## PRS 298, 305 et 311 recalibrate

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_test_eur_prs298_305_311_recalib.tsv"

df_prs = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

## Split de la cohorte européenne

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "train_ids_eur.csv"

train_ids_eur = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

In [ ]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "test_ids_eur.csv"

test_ids_eur = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

## Fusion des données

In [ ]:
df_merged = pd.merge(
    df_bcsc,
    df_mammorisk[['person_id','score_mammorisk','score_mammorisk_recalib','risk_group']],
    on='person_id',
    how='left'
)

In [ ]:
df_merged = pd.merge(
    df_merged,
    df_prs,
    on='person_id',
    how='left'
)

In [ ]:
df_merged.columns

In [ ]:
df_scores = df_merged[['person_id','has_bc',
                       'score_bcscRisk_recalib','score_mammorisk_recalib',
                       'prs_298','prs_305','prs_311',
                       'prs_298_z','prs_305_z','prs_311_z',
                       'prs_298_prob_recalib_only','prs_305_prob_recalib_only','prs_311_prob_recalib_only']]
df_scores

# TEST

In [ ]:
import numpy as np
import pandas as pd


def combine_clinical_and_prs(
    df_data,
    beta_prs=0.2789,  # Coefficient beta de la régression logistique du PRS-311
    bcsc_col="score_bcscRisk_recalib",
    mammo_col="score_mammorisk_recalib",
    prs_z_col="prs_311_z",
):
    """Combine les modèles cliniques (BCSC et Mammorisk) avec le PRS-311

    selon l'approche bayésienne/logistique à effets additifs (Élie Rassy et al.).
    """
    df_out = df_data.copy()

    # Protection contre log(0) ou log(1)
    eps = 1e-7
    p_bcsc = np.clip(df_out[bcsc_col], eps, 1 - eps)
    p_mammo = np.clip(df_out[mammo_col], eps, 1 - eps)

    # 1. Conversion des probabilités cliniques en log-odds (logit)
    logit_bcsc = np.log(p_bcsc / (1 - p_bcsc))
    logit_mammo = np.log(p_mammo / (1 - p_mammo))

    # 2. Addition du composant génétique (beta * PRS_z)
    genetic_component = beta_prs * df_out[prs_z_col]

    logit_bcsc_prs = logit_bcsc + genetic_component
    logit_mammo_prs = logit_mammo + genetic_component

    # 3. Reconversion inverse en probabilités absolues combinées
    df_out["prob_bcsc_prs311"] = 1.0 / (1.0 + np.exp(-logit_bcsc_prs))
    df_out["prob_mammorisk_prs311"] = 1.0 / (1.0 + np.exp(-logit_mammo_prs))

    return df_out

In [ ]:
# Application directe sur ton DataFrame df_scores
df_scores = combine_clinical_and_prs(
    df_data=df_scores,
    beta_prs=0.2741,  # Remplace par la valeur exacte de beta de ton PRS-311 si différente
    bcsc_col="score_bcscRisk_recalib",
    mammo_col="score_mammorisk_recalib",
    prs_z_col="prs_311_z",
)

In [ ]:
df_scores

In [ ]:
# Évaluation et comparaison des 5 modèles
df_summary = mf.evaluate_and_compare_all_models(
    df_data=df_scores,
    test_ids=test_ids_eur,
)

# Affichage du tableau récapitulatif
display(df_summary)

## Recalibration des modèles combinés

In [ ]:
# 1. Recalibration de l'intercept des modèles combinés
df_scores = mf.recalibrate_combined_models(
    df_data=df_scores,
    train_ids=train_ids_eur,
    test_ids=test_ids_eur,
    combined_cols=["prob_bcsc_prs311", "prob_mammorisk_prs311"],
)

# 2. Évaluation comparative intégrant les versions réétalonnées
models_to_evaluate = {
    "BCSC (Recalibré)": "score_bcscRisk_recalib",
    "Mammorisk (Recalibré)": "score_mammorisk_recalib",
    "PRS-311 (Seul)": "prs_311_prob_recalib_only",
    "BCSC + PRS-311 (Brut)": "prob_bcsc_prs311",
    "BCSC + PRS-311 (Ajusté)": "prob_bcsc_prs311_recalib",
    "Mammorisk + PRS-311 (Brut)": "prob_mammorisk_prs311",
    "Mammorisk + PRS-311 (Ajusté)": "prob_mammorisk_prs311_recalib",
}

df_summary_final = mf.evaluate_and_compare_all_models(
    df_data=df_scores, test_ids=test_ids_eur, models_dict=models_to_evaluate
)

display(df_summary_final)